# Setup
データ取り込みを行う。 DuckDB 想定。

In [ ]:
import os
import dotenv
from pathlib import Path
from sqlalchemy import create_engine

dotenv.load_dotenv()
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# 保存先ディレクトリを作成
vault_db_path = Path("../../tests/data/vault_db")
vault_db_path.mkdir(exist_ok=True)
vault_db_path = vault_db_path.resolve()

path = vault_db_path / "entity.duckdb"
sa_engine = create_engine(f"duckdb:///{path}")

# Extract

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
from assistant_agent.loaders import MarkdownReader

# Vault から読み込み
vault_path = Path("../../docs/dataset_website").resolve()
loader = SimpleDirectoryReader(
    input_dir=vault_path,
    recursive=True,
    file_extractor={".md": MarkdownReader()}
)
docs = loader.load_data()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
from sqlalchemy import create_engine, text
from assistant_agent.entities.duckdb import VaultBase, SampleEntity
from assistant_agent.entities.base import VaultUtils

# DB へ取り込み
with sa_engine.connect() as sess:
    sess.execute(text("create schema if not exists assets;"))
    sess.commit()

VaultBase.metadata.create_all(sa_engine)
VaultUtils.sync(docs[:3], sa_engine, SampleEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
import sqlalchemy
from pprint import pprint
from assistant_agent.entities.duckdb import SampleEntity

with sa_engine.connect() as sess:
    # sess.execute(text("checkpoint"))
    res = sess.execute(sqlalchemy.select(SampleEntity).limit(10)).all()
    pprint(res)

In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TransformComponent

chunk_size = 800
chunk_overlap = 80
JAPANESE_PARAGRAPH_SEP = "\n\n"

trans: list[TransformComponent] = [
    SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator=JAPANESE_PARAGRAPH_SEP,
    ),
]
pipe = IngestionPipeline(transformations=trans)
res = pipe.run(documents=docs[:2])
print("\n=====================\n".join([item.text for item in res]))  # pyright: ignore[reportAttributeAccessIssue]


In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

from assistant_agent.entities.duckdb import SampleEntity
from assistant_agent.services import VaultSampleRetriever
from assistant_agent.utils.store_context import DuckDBStoreContext

# レトリーバーを作成
store_ctx = DuckDBStoreContext(vault_db_path)
sa_engine = store_ctx.get_engine()
sample_retriever = VaultSampleRetriever(
    "sample_docstore",
    "sample_vectors",
    store_context=store_ctx,
    transformations=[
        SentenceSplitter(
            chunk_size=1024,
            chunk_overlap=200,
            paragraph_separator="\n\n",
        ),
    ],
    embed_model=GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=SampleEntity,
)

In [ ]:
# sample_retriever.sync_chunks()

# Retrieval

In [ ]:
query_res = sample_retriever.search_documents("人事・給与", 5)
for item in query_res:
    print(f"{item.score=}")  # pyright: ignore[reportAttributeAccessIssue]
    print(item.text)
    print("==================")